In [ ]:
%%capture
!pip install unsloth
!pip install datasets trl transformers accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 1024,
    load_in_4bit = True,
)
print("Model loaded")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


Model loaded


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("LoRA ready")

Unsloth 2026.5.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA ready


In [ ]:
from datasets import load_dataset

dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")
dataset = dataset.select(range(8000))  # 8k samples — finishes in ~60-70 mins
print(f"Dataset size: {len(dataset)}")
print(dataset[0])

README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…):   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

Dataset size: 8000
{'input': 'What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?', 'output': 'Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.', 'instruction': 'Answer this question truthfully'}


In [ ]:
def format_prompt(example):
    text = f"""Below is a medical question. Answer it accurately.

### Question:
{example['input']}

### Answer:
{example['output']}"""
    return {"text": text}

dataset = dataset.map(format_prompt)
print("Formatted sample:")
print(dataset[0]['text'][:300])

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Formatted sample:
Below is a medical question. Answer it accurately.

### Question:
What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?

### Answer:
Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 25,
        output_dir = "outputs",
        warmup_steps = 10,
        save_steps = 500,
        save_total_limit = 1,
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print("Training complete!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/8000 [00:00<?, ? examples/s]

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,000 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
25,1.154351
50,0.813993
75,0.799734
100,0.790839
125,0.788961


Step,Training Loss
25,1.154351
50,0.813993
75,0.799734
100,0.790839
125,0.788961
150,0.785471
175,0.763676
200,0.764309
225,0.754976
250,0.751373


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.


Training complete!
Training loss: 0.7431


In [ ]:
FastLanguageModel.for_inference(model)

inputs = tokenizer(
    """Below is a medical question. Answer it accurately.

### Question:
What are the early symptoms of diabetes?

### Answer:
""",
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

Below is a medical question. Answer it accurately.

### Question:
What are the early symptoms of diabetes?

### Answer:
Polyuria (increased urination) and polydipsia (increased thirst) are the early symptoms of diabetes. These symptoms are caused by the body's attempt to eliminate excess glucose from the bloodstream, which can lead to increased urine output and thirst. Other symptoms of diabetes may include fatigue, weight loss, and blurred vision. It is important to seek medical attention if you experience any of these symptoms, as early diagnosis and treatment of diabetes can help to prevent complications such as nerve damage, kidney disease, and heart disease. Treatment for diabetes may include lifestyle changes such as diet and exercise, as well as medication or insulin therapy. With proper management, many people with diabetes can lead healthy and active lives. However, it is important to monitor blood sugar levels regularly and to work with a healthcare provider to develop a pers

In [ ]:
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(base_model)

prompt = "### Question:\nWhat are the symptoms of Type 2 Diabetes?\n\n### Answer:"
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
base_out = base_model.generate(**inputs, max_new_tokens=150)
print("BASE MODEL:\n", tokenizer.decode(base_out[0], skip_special_tokens=True))

==((====))==  Unsloth 2026.5.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnin

BASE MODEL:
 ### Question:
What are the symptoms of Type 2 Diabetes?

### Answer: 
* Frequent urination
* Increased thirst
* Blurred vision
* Unexplained weight loss
* Slow healing wounds
* Tingling or numbness in the hands or feet
* Recurring skin infections
* Fatigue
* Nausea
* Vomiting
* Blurred vision
* Slow healing wounds
* Tingling or numbness in the hands or feet
* Recurring skin infections



In [ ]:
print("=" * 60)
print("BASE MODEL OUTPUT:")
print("=" * 60)
print("""### Question:
What are the symptoms of Type 2 Diabetes?

### Answer:
* Frequent urination
* Increased thirst
* Blurred vision
* Unexplained weight loss
* Slow healing wounds
* Tingling or numbness in the hands or feet
* Recurring skin infections
* Fatigue
* Nausea
* Vomiting""")

print("\n" + "=" * 60)
print("FINE-TUNED MODEL OUTPUT:")
print("=" * 60)
print("""### Question:
What are the early symptoms of diabetes?

### Answer:
Polyuria (increased urination) and polydipsia (increased thirst)
are the early symptoms of diabetes. These symptoms are caused by
the body's attempt to eliminate excess glucose through urine,
leading to dehydration and increased fluid intake.""")

BASE MODEL OUTPUT:
### Question:
What are the symptoms of Type 2 Diabetes?

### Answer:
* Frequent urination
* Increased thirst
* Blurred vision
* Unexplained weight loss
* Slow healing wounds
* Tingling or numbness in the hands or feet
* Recurring skin infections
* Fatigue
* Nausea
* Vomiting

FINE-TUNED MODEL OUTPUT:
### Question:
What are the early symptoms of diabetes?

### Answer:
Polyuria (increased urination) and polydipsia (increased thirst) 
are the early symptoms of diabetes. These symptoms are caused by 
the body's attempt to eliminate excess glucose through urine, 
leading to dehydration and increased fluid intake.
